## Descripción de Bagging (Bootstrap Aggregating)

**Bagging** (abreviatura de **Bootstrap Aggregating**) es una de las técnicas de *ensemble learning* más fundamentales y ampliamente utilizadas en el campo del Machine Learning. Su principal objetivo es mejorar la estabilidad y la precisión de los algoritmos predictivos, al tiempo que reduce la varianza y ayuda a prevenir el *overfitting*.

El funcionamiento de Bagging se basa en los siguientes principios:

1.  **Muestreo con reemplazo (Bootstrap):**
    *   El proceso comienza creando múltiples subconjuntos del dataset de entrenamiento original. A diferencia del muestreo tradicional, estos subconjuntos se forman seleccionando aleatoriamente observaciones del dataset original *con reemplazo*. Esto significa que una misma observación puede ser seleccionada múltiples veces para un subconjunto, o bien no ser seleccionada en absoluto para otros subconjuntos.
    *   Cada uno de estos subconjuntos, a menudo llamados "muestras bootstrap", tiene el mismo tamaño que el dataset de entrenamiento original, pero debido al muestreo con reemplazo, contendrán diferentes combinaciones de datos y algunas observaciones estarán duplicadas, mientras que otras estarán ausentes.

2.  **Entrenamiento de modelos base:**
    *   Para cada uno de los subconjuntos bootstrap generados, se entrena un modelo de aprendizaje individual. Estos modelos se conocen como "estimadores base" o "modelos base".
    *   Es crucial que los modelos base sean del mismo tipo (por ejemplo, todos árboles de decisión, todas máquinas de vectores de soporte (SVM), etc.).
    *   Al entrenar cada modelo en un subconjunto ligeramente diferente de los datos, se introduce una diversidad entre los modelos, ya que cada uno aprenderá patrones específicos de su propia muestra.

3.  **Agregación de predicciones:**
    *   Una vez que todos los modelos base han sido entrenados, sus predicciones individuales se combinan para obtener una predicción final más robusta.
    *   Para **tareas de clasificación**, la agregación se realiza típicamente mediante **votación mayoritaria**. Esto significa que la clase predicha por la mayoría de los modelos base se elige como la predicción final.
    *   Para **tareas de regresión**, las predicciones de los modelos base se combinan generalmente promediándolas.

### ¿Por qué funciona Bagging?

La efectividad de Bagging radica en su capacidad para reducir la **varianza** de un modelo. Los modelos con alta varianza (como los árboles de decisión no podados) son muy sensibles a pequeñas fluctuaciones en el conjunto de datos de entrenamiento y tienden a *overfittear*. Al promediar o votar las predicciones de múltiples modelos entrenados en diferentes vistas de los datos, Bagging suaviza estas fluctuaciones y produce un modelo final más estable y generalizable.

Aunque Bagging es excelente para reducir la varianza, no suele abordar el **sesgo** del modelo. Si los modelos base son intrínsecamente sesgados, el ensemble también lo será. Por esta razón, Bagging es particularmente efectivo cuando se utiliza con modelos base de alta varianza y bajo sesgo (como los árboles de decisión profundos o complejos), donde la reducción de la varianza es el factor clave para mejorar el rendimiento.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report, confusion_matrix

In [12]:
from sklearn.datasets import load_wine

# Cargar el dataset Wine
wine = load_wine()
X = wine.data  # Características
y = wine.target # Etiquetas de clase

# Convertir a DataFrame para una mejor visualización de las características
df_wine = pd.DataFrame(X, columns=wine.feature_names)
df_wine['target'] = y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) # Usamos stratify para mantener la proporción de clases

base_model = SVC(kernel='linear', random_state=42)

# Bagging
model = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    bootstrap=True,
    random_state=42
)

model.fit(X_train, y_train)

print("\nBagging entrenado exitosamente.")


# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar el rendimiento del modelo
mcc = matthews_corrcoef(y_test, y_pred)
print(f"MCC del modelo Bagging: {mcc:.4f}")

print("\nReporte de Clasificación:\n")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

print("\nMatriz de Confusión:\n")
print(confusion_matrix(y_test, y_pred))


Bagging entrenado exitosamente.
MCC del modelo Bagging: 0.9456

Reporte de Clasificación:

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       0.91      1.00      0.95        21
     class_2       1.00      0.87      0.93        15

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.97      0.96      0.96        54


Matriz de Confusión:

[[18  0  0]
 [ 0 21  0]
 [ 0  2 13]]
